# 05 - Original-image validation baseline

CPU only. Run after notebooks 02-04 on the same Colab runtime. This notebook evaluates only the locked validation split of FVC2000/2002 DB1-DB3. No FVC2004 or held-out test scores are computed.

Protocol: impression 1 is the fixed enrollment/gallery for every validation finger. Impressions 2-8 are probes. Genuine comparisons use the same finger; impostor comparisons use every other gallery finger within the same year/database. BOZORTH3 direction is always probe -> gallery. Future enhancement must alter probes only and reuse the pair lists and original galleries.

Preprocessing: grayscale; FVC2002 DB2 resampled from 569 to 500 dpi using Lanczos, other source DBs already 500 dpi; lossless PNG, no contrast enhancement. This is an original-image baseline with fixed resolution normalization, not an entirely unprocessed-byte baseline. Future methods must use the same normalized source pixels.

MINDTCT uses no -b or -m1. BOZORTH3 uses minminutiae=10 and its default 150 highest-quality minutiae. Templates with fewer than 10 points are explicit acquisition/extraction failures and reject every affected attempt; their scores remain missing, never fake zero scores. Nonzero process exits or malformed output abort the run with logs, rather than masquerading as quality failures.

Metrics per database: conditional FMR/FNMR among matchable pairs, approximate discrete conditional EER, and attempt-level FAR/FRR including rejected attempts. Choose the smallest nonnegative integer threshold with empirical conditional validation FMR <= 1%. This is calibration on validation, not an unbiased test estimate. Future methods calibrate their own thresholds on this same validation protocol, never on test.

Duplicates: primary results retain all official impressions; a secondary view keeps the first byte-distinct impression per scoped finger (including enrollment in the deduplication order). It applies the primary threshold without recalibration. Hash deduplication cannot detect transformed copies. Pair counts are correlated; do not infer statistical confidence from the number of pairs alone.

Pair lists and configuration are frozen before extracting any features. Outputs and progress are on Drive; local template cache permits rerunning after an interruption on the same runtime. Biometric templates and reports are sensitive research data: do not redistribute automatically.

In [ ]:
import hashlib
import json
import math
import re
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPORT_DIR = PROJECT_ROOT / 'baselines' / 'original_validation_v1' / RUN_ID
REPORT_DIR.mkdir(parents=True, exist_ok=True)
NBIS_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
MINDTCT = NBIS_ROOT / 'install' / 'bin' / 'mindtct'
BOZORTH3 = NBIS_ROOT / 'install' / 'bin' / 'bozorth3'

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True) + '\n').encode('utf-8')

def csv_bytes(frame):
    return frame.to_csv(index=False, lineterminator='\n').encode('utf-8')

split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
for name, expected in split_protocol['file_sha256'].items():
    if file_digest(SPLIT_DIR / name) != expected:
        raise RuntimeError(f'Frozen split file changed: {name}')
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
validation = locked[locked['split'].eq('validation')].sort_values(
    ['dataset', 'db', 'finger_id', 'impression_id']).reset_index(drop=True)
if validation.empty or not set(validation['dataset']).issubset({'FVC2000', 'FVC2002'}):
    raise RuntimeError('Invalid validation split.')
stamp_path = NBIS_ROOT / 'verified_build.json'
if not stamp_path.is_file():
    raise FileNotFoundError('NBIS runtime installation missing. Run notebook 04 first.')
build = json.loads(stamp_path.read_text(encoding='utf-8'))
if build['signature']['source_sha256'] != '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3':
    raise RuntimeError('Unexpected NBIS source version.')
for binary in (MINDTCT, BOZORTH3):
    if not binary.is_file() or file_digest(binary) != build['binary_sha256'][binary.name]:
        raise RuntimeError(f'NBIS binary missing or modified: {binary}')
    subprocess.run([str(binary), '-version'], check=True, capture_output=True, text=True)
print('Validation images:', len(validation))
print('Reports:', REPORT_DIR)

In [ ]:
def make_pairs(frame):
    rows = []
    for (dataset, db), group in frame.groupby(['dataset', 'db'], sort=True):
        group = group.sort_values(['identity_key', 'impression_id'])
        if group.duplicated(['identity_key', 'impression_id']).any():
            raise RuntimeError('Duplicate identity/impression keys in validation.')
        for identity, finger in group.groupby('identity_key'):
            if set(finger['impression_id']) != set(range(1, 9)):
                raise RuntimeError(f'Expected impressions 1-8: {identity}')
        gallery = group[group['impression_id'].eq(1)].to_dict('records')
        probes = group[~group['impression_id'].eq(1)].to_dict('records')
        distinct_paths = set(group.drop_duplicates(['identity_key', 'sha256'])['relative_path'])
        if len(gallery) < 2:
            raise RuntimeError(f'Need at least two validation fingers: {dataset} {db}')
        for probe in probes:
            for enrollment in gallery:
                same = probe['identity_key'] == enrollment['identity_key']
                pair_key = probe['relative_path'] + '|' + enrollment['relative_path']
                rows.append({
                    'pair_id': hashlib.sha256(pair_key.encode('utf-8')).hexdigest(),
                    'dataset': dataset, 'db': db, 'label': int(same),
                    'probe_path': probe['relative_path'], 'gallery_path': enrollment['relative_path'],
                    'probe_identity': probe['identity_key'], 'gallery_identity': enrollment['identity_key'],
                    'duplicate_sensitivity_eligible': (
                        probe['relative_path'] in distinct_paths and enrollment['relative_path'] in distinct_paths),
                })
    result = pd.DataFrame(rows)
    if result['pair_id'].duplicated().any():
        raise RuntimeError('Duplicate pair IDs.')
    return result

pairs = make_pairs(validation)
DPI = {
    'FVC2000_DB1': 500, 'FVC2000_DB2': 500, 'FVC2000_DB3': 500,
    'FVC2002_DB1': 500, 'FVC2002_DB2': 569, 'FVC2002_DB3': 500,
}
configuration = {
    'version': 'nbis_validation_pairs_v1',
    'split_protocol_sha256': file_digest(SPLIT_DIR / 'protocol.json'),
    'pair_csv_sha256': hashlib.sha256(csv_bytes(pairs)).hexdigest(),
    'gallery': 'impression 1; unchanged original after fixed normalization',
    'probes': 'impressions 2-8; each against every gallery in same dataset/db',
    'normalization': 'L-mode PNG, Lanczos to 500 dpi, no enhancement',
    'source_dpi': DPI, 'pillow_version': Image.__version__,
    'mindtct_flags': [], 'bozorth3_flags': ['-A', 'minminutiae=10'],
    'bozorth3_default_max_minutiae': 150,
    'failure_policy': 'fewer than 10 minutiae => reject affected attempts, score missing; process errors abort',
    'threshold_rule': 'smallest nonnegative integer with conditional validation FMR <= 0.01; accept score >= threshold',
    'duplicate_sensitivity': 'first byte-distinct impression per scoped finger; same primary threshold',
    'nbis_source_sha256': build['signature']['source_sha256'],
}
frozen_files = {'pairs.csv': csv_bytes(pairs), 'configuration.json': json_bytes(configuration)}
PAIR_DIR.mkdir(parents=True, exist_ok=True)
for name, data in frozen_files.items():
    path = PAIR_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen pair configuration differs: {path}. Do not overwrite it.')
for name, data in frozen_files.items():
    path = PAIR_DIR / name
    if not path.exists():
        with path.open('xb') as output:
            output.write(data)
    if path.read_bytes() != data:
        raise RuntimeError(f'Pair protocol persistence failed: {path}')
pair_summary = pairs.groupby(['dataset', 'db', 'label']).size().unstack(fill_value=0)
pair_summary.columns = ['impostor_pairs', 'genuine_pairs']
print(pair_summary.to_string())
print('Pair protocol frozen before feature extraction:', PAIR_DIR)

In [ ]:
CACHE_TAG = hashlib.sha256(json_bytes({
    'configuration': configuration, 'binary_sha256': build['binary_sha256'],
})).hexdigest()[:20]
CACHE_DIR = LOCAL_ROOT / 'validation_templates' / CACHE_TAG
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def read_xyt(path):
    points = []
    for line in path.read_text(encoding='ascii').splitlines():
        if line.strip():
            values = [int(value) for value in line.split()]
            if len(values) != 4:
                raise RuntimeError(f'Malformed MINDTCT row: {path}: {line}')
            points.append(values)
    return points

def extract_template(record):
    source = DATA_ROOT / record['relative_path']
    if not source.is_file():
        raise FileNotFoundError('Runtime images missing. Rerun notebook 02 to restore them.')
    if file_digest(source) != record['sha256']:
        raise RuntimeError(f'Image differs from audit: {source}')
    key = hashlib.sha256(record['relative_path'].encode('utf-8')).hexdigest()
    root = CACHE_DIR / key
    xyt = root.with_suffix('.xyt')
    metadata = root.with_suffix('.json')
    if metadata.is_file():
        cached = json.loads(metadata.read_text(encoding='utf-8'))
        if (cached['source_sha256'] != record['sha256'] or not xyt.is_file()
                or file_digest(xyt) != cached['template_sha256']):
            raise RuntimeError(f'Cached extraction integrity failure: {metadata}')
        return cached
    dpi = DPI[f"{record['dataset']}_{record['db']}"]
    png = root.with_suffix('.png')
    with Image.open(source) as image:
        normalized = image.convert('L')
        if dpi != 500:
            size = tuple(max(1, round(length * 500 / dpi)) for length in normalized.size)
            normalized = normalized.resize(size, resample=Image.Resampling.LANCZOS)
        normalized.save(png, dpi=(500, 500))
        width, height = normalized.size
    result = subprocess.run([str(MINDTCT), str(png), str(root)],
                            capture_output=True, text=True, timeout=120)
    root.with_suffix('.log').write_text(result.stdout + result.stderr, encoding='utf-8')
    if result.returncode or not xyt.is_file():
        raise RuntimeError(f'MINDTCT process failed for {source}: code {result.returncode}\n{result.stderr}')
    points = read_xyt(xyt)
    row = {
        'relative_path': record['relative_path'], 'dataset': record['dataset'], 'db': record['db'],
        'identity_key': record['identity_key'], 'impression_id': int(record['impression_id']),
        'source_sha256': record['sha256'], 'source_dpi': dpi, 'width': width, 'height': height,
        'minutiae': len(points), 'status': 'ok' if len(points) >= 10 else 'insufficient_minutiae',
        'xyt_path': str(xyt), 'template_sha256': file_digest(xyt),
    }
    metadata.write_bytes(json_bytes(row))
    return row

extraction_rows = []
for index, record in enumerate(validation.to_dict('records'), start=1):
    extraction_rows.append(extract_template(record))
    if index % 50 == 0 or index == len(validation):
        pd.DataFrame(extraction_rows).to_csv(REPORT_DIR / 'extraction_progress.csv', index=False)
        print(f'Extracted/cached {index}/{len(validation)} images', flush=True)
extraction = pd.DataFrame(extraction_rows)
extraction.to_csv(REPORT_DIR / 'extraction.csv', index=False)
with zipfile.ZipFile(REPORT_DIR / 'templates_and_logs.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for record in extraction_rows:
        root = Path(record['xyt_path']).with_suffix('')
        for extension in ('.xyt', '.json', '.log'):
            path = root.with_suffix(extension)
            if path.is_file():
                archive.write(path, arcname=path.name)
print(extraction.groupby(['dataset', 'db', 'status']).size().to_string())
print('All low-minutiae images remain in the attempt denominators.')

In [ ]:
templates = extraction.set_index('relative_path')
scored = pairs.copy()
scored['probe_status'] = scored['probe_path'].map(templates['status'])
scored['gallery_status'] = scored['gallery_path'].map(templates['status'])
if scored[['probe_status', 'gallery_status']].isna().any().any():
    raise RuntimeError('Pair references an image missing from extraction.')
scored['matchable'] = scored['probe_status'].eq('ok') & scored['gallery_status'].eq('ok')
scored['score'] = np.nan
valid_indices = scored.index[scored['matchable']].tolist()
BATCH_SIZE = 256
for offset in range(0, len(valid_indices), BATCH_SIZE):
    indices = valid_indices[offset:offset + BATCH_SIZE]
    batch = scored.loc[indices]
    mates = CACHE_DIR / 'current_mates.lis'
    lines = []
    for pair in batch.to_dict('records'):
        lines.extend([templates.loc[pair['probe_path'], 'xyt_path'],
                      templates.loc[pair['gallery_path'], 'xyt_path']])
    mates.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    result = subprocess.run([str(BOZORTH3), '-A', 'minminutiae=10', '-M', str(mates)],
                            capture_output=True, text=True, timeout=600)
    output_lines = result.stdout.splitlines()
    log_path = REPORT_DIR / f'match_batch_{offset // BATCH_SIZE:04d}.log'
    log_path.write_text(result.stdout + '\nSTDERR:\n' + result.stderr, encoding='utf-8')
    if (result.returncode or len(output_lines) != len(indices)
            or any(re.fullmatch(r'\s*\d+\s*', line) is None for line in output_lines)):
        raise RuntimeError(f'BOZORTH3 batch failed or output count/format invalid. See {log_path}')
    scored.loc[indices, 'score'] = [int(line.strip()) for line in output_lines]
    if (offset // BATCH_SIZE + 1) % 10 == 0 or offset + len(indices) == len(valid_indices):
        scored.to_csv(REPORT_DIR / 'scores_progress.csv', index=False)
    print(f'Matched {offset + len(indices)}/{len(valid_indices)} valid pairs', flush=True)
if scored.loc[scored['matchable'], 'score'].isna().any():
    raise RuntimeError('Missing score for a matchable pair.')
scored.to_csv(REPORT_DIR / 'scores.csv', index=False)
print('Pairs rejected due to insufficient minutiae:', (~scored['matchable']).sum())

In [ ]:
def calibrate_threshold(impostor_scores, target=0.01):
    values = np.asarray(impostor_scores, dtype=float)
    if values.size == 0 or not np.isfinite(values).all():
        raise RuntimeError('Cannot calibrate without finite, matchable impostor scores.')
    if not np.equal(values, np.floor(values)).all() or (values < 0).any() or not 0 <= target < 1:
        raise ValueError('Expected nonnegative integer similarity scores and target in [0,1).')
    allowed = math.floor(target * len(values) + 1e-12)
    return int(np.sort(values)[::-1][allowed]) + 1

def conditional_curve(frame):
    valid = frame[frame['matchable']]
    genuine = np.sort(valid.loc[valid['label'].eq(1), 'score'].to_numpy(dtype=float))
    impostor = np.sort(valid.loc[valid['label'].eq(0), 'score'].to_numpy(dtype=float))
    if not len(genuine) or not len(impostor):
        raise RuntimeError('Conditional metrics need matchable genuine and impostor pairs.')
    thresholds = np.unique(np.concatenate(([0.0], genuine + 1, impostor + 1)))
    return pd.DataFrame({
        'threshold': thresholds,
        'conditional_fmr': (len(impostor) - np.searchsorted(impostor, thresholds, side='left')) / len(impostor),
        'conditional_fnmr': np.searchsorted(genuine, thresholds, side='left') / len(genuine),
    })

def metrics_at_threshold(frame, threshold):
    if frame.empty or not set(frame['label']).issubset({0, 1}):
        raise ValueError('Expected nonempty binary-labelled pair table.')
    if frame.loc[frame['matchable'], 'score'].isna().any():
        raise RuntimeError('Matchable pair has no score.')
    accepted = frame['matchable'] & frame['score'].ge(threshold)
    genuine = frame['label'].eq(1)
    impostor = ~genuine
    valid_genuine = genuine & frame['matchable']
    valid_impostor = impostor & frame['matchable']
    if not genuine.any() or not impostor.any():
        raise RuntimeError('Both genuine and impostor attempts are required.')
    return {
        'threshold': int(threshold), 'genuine_attempts': int(genuine.sum()),
        'impostor_attempts': int(impostor.sum()),
        'unmatchable_genuine': int((genuine & ~frame['matchable']).sum()),
        'unmatchable_impostor': int((impostor & ~frame['matchable']).sum()),
        'attempt_far': float(accepted[impostor].mean()),
        'attempt_frr': float((~accepted[genuine]).mean()),
        'conditional_fmr': float(accepted[valid_impostor].mean()) if valid_impostor.any() else None,
        'conditional_fnmr': float((~accepted[valid_genuine]).mean()) if valid_genuine.any() else None,
    }

metric_rows = []
threshold_rows = []
curves = []
for (dataset, db), group in scored.groupby(['dataset', 'db'], sort=True):
    values = group.loc[group['matchable'] & group['label'].eq(0), 'score']
    threshold = calibrate_threshold(values, target=0.01)
    curve = conditional_curve(group)
    closest = (curve['conditional_fmr'] - curve['conditional_fnmr']).abs().idxmin()
    point = curve.loc[closest]
    primary = metrics_at_threshold(group, threshold)
    if primary['conditional_fmr'] > 0.01 + 1e-12:
        raise RuntimeError('Calibration did not meet empirical validation FMR target.')
    metric_rows.append({
        'dataset': dataset, 'db': db, 'view': 'all_impressions', **primary,
        'conditional_eer_discrete_approx': float((point['conditional_fmr'] + point['conditional_fnmr']) / 2),
        'eer_crossing_gap': float(abs(point['conditional_fmr'] - point['conditional_fnmr'])),
    })
    sensitivity = group[group['duplicate_sensitivity_eligible']]
    metric_rows.append({'dataset': dataset, 'db': db, 'view': 'byte_distinct_sensitivity',
                        **metrics_at_threshold(sensitivity, threshold)})
    threshold_rows.append({'dataset': dataset, 'db': db, 'threshold': threshold,
                           'calibrated_on': 'original validation matchable impostor pairs',
                           'empirical_fmr': primary['conditional_fmr']})
    curves.append(curve.assign(dataset=dataset, db=db))
metrics = pd.DataFrame(metric_rows)
thresholds = pd.DataFrame(threshold_rows)
metrics.to_csv(REPORT_DIR / 'validation_metrics.csv', index=False)
thresholds.to_csv(REPORT_DIR / 'validation_thresholds.csv', index=False)
pd.concat(curves, ignore_index=True).to_csv(REPORT_DIR / 'conditional_curves.csv', index=False)
failure_summary = extraction.groupby(['dataset', 'db']).agg(
    images=('relative_path', 'size'),
    insufficient_minutiae=('status', lambda values: int(values.ne('ok').sum())),
).reset_index()
failure_summary['image_failure_rate'] = failure_summary['insufficient_minutiae'] / failure_summary['images']
failure_summary.to_csv(REPORT_DIR / 'image_failure_rates.csv', index=False)
print(metrics.to_string(index=False))
print('\nImage failures:\n', failure_summary.to_string(index=False))
print('Rates are fractions (0.10 = 10%). Validation calibration is not a test result.')

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(14, 8))
groups = list(scored.groupby(['dataset', 'db'], sort=True))
if len(groups) != 6:
    raise RuntimeError('Expected six validation database strata.')
for axis, ((dataset, db), group) in zip(axes.flat, groups):
    for label, name in ((1, 'genuine'), (0, 'impostor')):
        values = group.loc[group['matchable'] & group['label'].eq(label), 'score']
        axis.hist(values, bins=40, alpha=0.5, density=True, label=name)
    axis.set_title(f'{dataset} {db}')
    axis.set_xlabel('BOZORTH3 similarity (matchable pairs only)')
    axis.set_ylabel('Density')
    axis.legend()
figure.tight_layout()
figure.savefig(REPORT_DIR / 'validation_score_distributions.png', dpi=150)
plt.show()
completion = {
    'status': 'validation_baseline_complete', 'run_id': RUN_ID,
    'pair_configuration_sha256': file_digest(PAIR_DIR / 'configuration.json'),
    'nbis_binary_sha256': build['binary_sha256'],
    'evaluated_split': 'validation', 'evaluated_external_test': False,
    'score_rows': len(scored), 'validation_images': len(extraction),
    'report_hashes': {path.name: file_digest(path) for path in REPORT_DIR.iterdir() if path.is_file()},
}
(REPORT_DIR / 'completion.json').write_bytes(json_bytes(completion))
print('ORIGINAL VALIDATION BASELINE COMPLETE:', REPORT_DIR)
print('Save notebook outputs. Next: review failures and lock synthetic degradation design using development data only.')